We have our education and income data by precinct. However, our naming scheme is not consisent - some precincts have numbers, some have titles, some have extra long names. We're going to fix this now!

If our data were more consisent, we could do this by text. However, we are going to do this by geometry using Geopandas.

In [13]:
# Install the necessary packages
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from shapely.validation import make_valid
BASE_DIR = Path().resolve().parent.parent # So that we can access our data outside of the Code folder

In [23]:
# Import the necessary files
voting_precints_gdf = gpd.read_file(BASE_DIR / "Shapefile_Data" / "Voting_Precincts" / "Voting_Precincts.shp") # Our master list of precinct geometries, names, and numbers
nc_2022_gdf = gpd.read_file(BASE_DIR / "Shapefile_Data" / "NC_2022_Precincts_Education_income" / "NC_2022_Precincts_Education_income.shp") # 2022 data
nc_2023_gdf = gpd.read_file(BASE_DIR / "Shapefile_Data" / "NC_2023_Precincts_Education_income" / "NC_2023_Precincts_Education_income.shp") # 2023 data

In [37]:
voting_precints_2022_gdf = gpd.read_file(BASE_DIR / "Shapefile_Data/SBE_PRECINCTS_CENSUSBLOCKS_20220118/SBE_PRECINCTS_CENSUSBLOCKS_20220118.shp")

In [ ]:
# Clean up the voting precincts gdf
voting_precints_gdf = voting_precints_gdf.to_crs("EPSG:26915")
voting_precints_gdf = voting_precints_gdf[['county_id', 'prec_id', 'enr_desc', 'county_nam', 'geometry']]

In [40]:
voting_precints_2022_gdf.nunique()

id            236638
county_id        100
prec_id         1931
geoid20       236638
countyfp20       100
tractce20       1776
blockce20       1187
update_dt          4
enr_desc        2543
county_nam       100
to_enr_des       290
to_prec_id       299
geometry      236638
dtype: int64

In [38]:
# Use an sjoin to merge for 2022
precinct_2022_merged = gpd.sjoin(nc_2022_gdf, voting_precints_2022_gdf)
# Do the same thing to merge for 2023
precinct_2022_merged = gpd.sjoin(nc_2023_gdf, voting_precints_gdf)

/var/folders/tx/qckl6gp57yq_ckc0ywsl5vlh0000gn/T/ipykernel_36903/2068518659.py:2: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:26915
Right CRS: EPSG:2264

  precinct_2022_merged = gpd.sjoin(nc_2022_gdf, voting_precints_2022_gdf)


In [30]:
precinct_2022_merged

,COUNTYNM,PRECINCT,total_vap,under_9g,9_to_12_nd,hs_grad,scol_nd,assoc_deg,bach_deg,grad_deg_p,...,objectid,id,county_id,prec_id,enr_desc,county_nam,of_prec_id,shape_leng,st_areasha,st_perimet
0,BURKE,0001,4015.0,120.0,274.0,1067.0,1144.0,470.0,722.0,218.0,...,2,2,12,0003,DREXEL 03,BURKE,None,49918.849040,1.083199e+08,49918.849040
0,BURKE,0001,4015.0,120.0,274.0,1067.0,1144.0,470.0,722.0,218.0,...,2209,9999,12,0001,DREXEL 01,BURKE,None,73531.396007,2.258691e+08,73531.396007
0,BURKE,0001,4015.0,120.0,274.0,1067.0,1144.0,470.0,722.0,218.0,...,41,30,12,0021,LOVELADY 01,BURKE,None,54729.875194,9.786835e+07,54729.875194
0,BURKE,0001,4015.0,120.0,274.0,1067.0,1144.0,470.0,722.0,218.0,...,46,35,12,0024,LOVELADY 04,BURKE,None,116919.118962,2.521890e+08,116919.118962
0,BURKE,0001,4015.0,120.0,274.0,1067.0,1144.0,470.0,722.0,218.0,...,55,49,12,0038,MORGANTON 08,BURKE,None,86648.627575,2.079628e+08,86648.627576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2654,CASWELL,YANCEYVILLE,2925.0,95.0,487.0,1133.0,378.0,352.0,363.0,117.0,...,1800,2023,17,LEAS,LEASBURG,CASWELL,None,150172.735523,9.612347e+08,150172.735523
2654,CASWELL,YANCEYVILLE,2925.0,95.0,487.0,1133.0,378.0,352.0,363.0,117.0,...,1795,2038,17,LOCU,LOCUST HILL,CASWELL,None,216010.441894,1.385679e+09,216010.441894
2654,CASWELL,YANCEYVILLE,2925.0,95.0,487.0,1133.0,378.0,352.0,363.0,117.0,...,2040,2265,17,PELH,PELHAM,CASWELL,None,244131.241726,1.757478e+09,244131.241726
2654,CASWELL,YANCEYVILLE,2925.0,95.0,487.0,1133.0,378.0,352.0,363.0,117.0,...,2113,2361,17,PROVI,PROVIDENCE,CASWELL,None,103716.101700,4.386818e+08,103716.101700


In [24]:
from enum import Enum

class Crs(Enum):
    """
    Convenience class for storing default CRS
    """
    WGS84 = "epsg:4326"
    UsaContigAlbersEqAreaConic = "epsg:102003"

In [25]:
# Code copied from https://gist.github.com/Greenall/2ac7071a4aa1cc165e2bbd4c5009092e, who publicly posed this for use.
def sjoin_greatest_intersection(
    df1: gpd.GeoDataFrame,
    df2: gpd.GeoDataFrame,
    groupby: list,
    rsuffix: str = "right",
    how: str = "inner",
):
    """
    Performs an sjoin between df1 and df2 to find intersecting shapes.
    For shapes in df1 that intersect more than one shape in df2,
    retain only the match with the greatest intersection.
    Note this is different to standard sjoin behaviour whereby
    you get a row for every shape that intersects.
    Args:
      df1 : source geodataframe
      df2 : df to join
      groupby: list of columns that form the index. will get max
      one row per unique (exactly one if left join)
      rsuffix: suffix to apply to overlapping col names in the rhs
      how: inner or left join supported
    """
    if isinstance(groupby, str):
        groupby = [groupby]
    if any(col in df2 for col in groupby):
        raise ValueError(
            "Overlapping groupby in df1 and df2"
            "drop overlapping cols from df2 before sjoin."
        )

    # keep the original geom as the projection for area calc
    # is not safely reversible
    original_geom = df1[["geometry", *groupby]].set_index(groupby)
    area_preserving_crs = Crs.UsaContigAlbersEqAreaConic.value

    df2 = df2.to_crs(area_preserving_crs)
    df1 = df1.to_crs(area_preserving_crs)
    # transformation to projected geometry can cause some shapes
    # to become invalid. force validity here
    df1.geometry = df1.geometry.apply(make_valid)
    df2.geometry = df2.geometry.apply(make_valid)

    # for some reason sjoin behaves differently to pandas.join in that
    # if there are overlapping colnames and you supply rsuffix but not
    # lsuffix then you still get a trailing underscore appended
    # to overlapping left column names. use arbitrary suffix here so we
    # can safely strip later
    lsuffix="somethingarbitrary"
    initial_join = df1.sjoin(
        df2, predicate="intersects", lsuffix=lsuffix, rsuffix=rsuffix, how=how
    )


    def select_max_intersect(df2):
        def inner(grp):
            # if more than one intersection have to work out best
            if grp.shape[0] > 1:
                intersections = grp.intersection(
                    df2.loc[grp[f"index_{rsuffix}"]], align=False
                )
                grp = grp.iloc[np.argsort(intersections.area.values)[-1]]

            else:
                grp = grp.iloc[0]
            return grp.drop(f"index_{rsuffix}")

        return inner

    result = initial_join.groupby(groupby).apply(select_max_intersect(df2))
    # restore the original geometry
    result.drop("geometry", axis=1, inplace=True)
    result = result.join(original_geom)
    result.columns = [c.replace(f"_{lsuffix}", "") for c in result.columns]
    result.crs = original_geom.crs

    return result

In [17]:
nc_2022_gdf.columns

Index(['COUNTYNM', 'PRECINCT', 'total_vap', 'under_9g', '9_to_12_nd',
       'hs_grad', 'scol_nd', 'assoc_deg', 'bach_deg', 'grad_deg_p', 'less_10k',
       '10k_15k', '15k_20k', '20k_25k', '25k_30k', '30k_35k', '35k_40k',
       '40k_45k', '45k_50k', '50k_60k', '60k_75k', '75k_100k', '100k_125k',
       '125k_150k', '150k_200k', '200k_plus', 'geometry'],
      dtype='object')

In [26]:
sjoin_greatest_intersection(nc_2022_gdf, voting_precints_gdf, "PRECINCT")

CRSError: Invalid projection: epsg:102003: (Internal Proj Error: proj_create: crs not found: EPSG:102003)

In [19]:
print(type(nc_2022_gdf))
print(type(voting_precints_gdf))

<class 'geopandas.geodataframe.GeoDataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [ ]:
from shapely import wkt

df["Coordinates"] = geopandas.GeoSeries.from_wkt(df["Coordinates"])